In [62]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt


--2026-05-23 16:38:49--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.22’

input.txt.22        100%[===================>]   1.06M  --.-KB/s    in 0.009s  

2026-05-23 16:38:49 (121 MB/s) - ‘input.txt.22’ saved [1115394/1115394]



In [63]:
with open("input.txt", encoding="utf-8") as f:
    text = f.read()

In [64]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)
# Hyper-parameters
batch_size = 64
block_size = 256
max_iters = 5000
eval_interval = 300
learning_rate = 3e-4
head_size = 16
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)
eval_iters = 200
n_embd = 384
head_size = n_embd
dropout = 0.2



cuda


In [65]:
# Unuque characters in our dataset
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))
print(vocab_size)



 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [66]:
# encode and decode popular ones (sentencePiece)

stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print (encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [67]:
# encode our dataset
data = torch.tensor(encode(text), dtype = torch.long)
print (data.shape, data.dtype)
print (data[:100])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59])


In [68]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

In [69]:
torch.manual_seed(1337)
# data loader
def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x,y = x.to(device), y.to(device)
    return x, y

xb, yb = get_batch("train")
print("inputs:")
print(xb.shape)
print(xb)
print("targets:")
print(yb.shape)
print(yb)

print("----")

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b,t]
        print (f"when input is {context.tolist()} the target: {target}")


inputs:
torch.Size([64, 256])
tensor([[ 0, 26, 53,  ..., 56, 43, 47],
        [60, 43, 56,  ..., 56,  1, 41],
        [26, 21, 33,  ..., 26, 21, 13],
        ...,
        [ 5, 57,  1,  ...,  1, 35, 47],
        [56, 53, 53,  ..., 59, 50, 42],
        [42, 47, 56,  ..., 39, 56,  1]], device='cuda:0')
targets:
torch.Size([64, 256])
tensor([[26, 53, 58,  ..., 43, 47, 45],
        [43, 56,  1,  ...,  1, 41, 53],
        [21, 33, 31,  ..., 21, 13, 10],
        ...,
        [57,  1, 52,  ..., 35, 47, 50],
        [53, 53, 58,  ..., 50, 42,  1],
        [47, 56, 43,  ..., 56,  1, 51]], device='cuda:0')
----
when input is [0] the target: 26
when input is [0, 26] the target: 53
when input is [0, 26, 53] the target: 58
when input is [0, 26, 53, 58] the target: 1
when input is [0, 26, 53, 58, 1] the target: 19
when input is [0, 26, 53, 58, 1, 19] the target: 50
when input is [0, 26, 53, 58, 1, 19, 50] the target: 53
when input is [0, 26, 53, 58, 1, 19, 50, 53] the target: 59
when input is [0, 26,

In [70]:
class Head (nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias = False)
        self.query = nn.Linear(n_embd, head_size, bias = False)
        self.value = nn.Linear(n_embd, head_size, bias = False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))
        self.dropout = nn.Dropout(dropout)
    
    def forward(self,x):
        B,T,C = x.shape
        
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei  = q @ k.transpose(-2,-1) * C**-0.5
        wei  = wei.masked_fill(self.tril[:T,:T] == 0 , float('-inf'))
        wei = F.softmax(wei, dim =-1)
        wei = self.dropout(wei)
        
        out = wei @ v
        return out


In [71]:
class MultiHeadedAttension(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.projection = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)


    def forward (self, x):
        out =  torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.projection(out)
        return out

In [72]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)


In [73]:
# Transformer block
class Block(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadedAttension(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward (self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [74]:
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(Block(n_embd,n_head=4), Block(n_embd,n_head=4), Block(n_embd,n_head=4),)
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward (self, idx, targets = None):
        B, T  = idx.shape
        token_embeddings = self.token_embedding_table(idx)
        position_embeddings = self.position_embedding_table(torch.arange(T, device=device))
        x = token_embeddings + position_embeddings
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate (self, idx, max_new_tokens):
        for _ in range (max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits [:, -1, :]
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx,idx_next), dim=1)
        return idx


model = BigramLanguageModel()
m = model.to(device)
logits, loss = m(xb,yb)
print(logits.shape)
print(loss)

torch.Size([16384, 65])
tensor(4.3230, device='cuda:0', grad_fn=<NllLossBackward0>)


In [75]:
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

In [77]:
for steps in range(max_iters):
    xb, yb = get_batch("train")

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % eval_interval == 0:
        print(f"step {steps}: loss {loss.item():.4f}")

print(loss.item())


step 0: loss 1.2713
step 300: loss 1.2840
step 600: loss 1.2416


KeyboardInterrupt: 

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device = device)
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))




NCER:
BRAMNTIO:
Nentis the the a selk Bay the andgrad my dagataus!
Yourthafuly he herty cedtlancanesswack my than dack zong heaouns, toffitie bettlit nowlincees is ensengmin;
Stiselidrove the the know him.

Wans!
 likind tear.
-hell courvey: the hather, why.

KING HIRDBET:
Yell I whom
the the lamake onWing thurt evings the my murish cenchimeds pood

And The
he kidly,
Ture fir the grean whithen male of which Pried my of thath suk!

JOMPUY:

Sadake as have stavein courrear tey Rry the hands care,
